**PS5 — Remaining Useful Life (Survival Analysis)**

---
SageMaker Studio | MLflow | Feature Store | Model Registry | Model Monitor

Architecture  : CUBIC MARS AWS v6
Target        : days_to_failure (regression) + is_censored (binary)
Grain         : one row per DEVICE_ID per component (TVM/GATE/READER/VALIDATOR)
Models        : WeibullAFT | CoxPH | RandomSurvivalForest | KM curves
Primary metric: concordance index (C-index), NOT AUC
Feasibility   : 50%
  Gap 1       : No work order (WO) data — PM/CM intervals unknown
  Gap 2       : No PM data for GATE/READER/VALIDATOR (TVM only partial via SERVICENOW)
  Gap 3       : No component-level failure timestamps → label from last seen event

---

In [ ]:
# ── CELL 1 : Install ──────────────────────────────────────────────────────────
import subprocess, sys
pkgs = [
    "s3fs","pyarrow",
    "lifelines>=0.29","scikit-survival>=0.23","numba>=0.58",
    "scikit-learn>=1.4","xgboost>=2.0",
    "optuna>=3.6","shap>=0.45","mlflow>=2.13", "sagemaker-mlflow",
    "boto3","sagemaker>=2.220,<3.0","pandas>=2.0",
    "matplotlib","seaborn","scipy","joblib",
]
subprocess.check_call([sys.executable,"-m","pip","install","-q","--upgrade"]+pkgs)
print("OK")

In [ ]:
# ── CELL 2 : Imports ──────────────────────────────────────────────────────────
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap, optuna, boto3, sagemaker, joblib, tarfile

# deltalake removed: gold tables exported as Parquet; pd.read_parquet() used in CELL 5

# Survival analysis
from lifelines import (WeibullAFTFitter, CoxPHFitter, KaplanMeierFitter,
                        NelsonAalenFitter)
from lifelines.utils import concordance_index
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv

# ML / monitoring
from sklearn.model_selection    import TimeSeriesSplit
from sklearn.preprocessing      import StandardScaler
from sklearn.ensemble           import GradientBoostingRegressor
from sklearn.metrics            import mean_absolute_error
import mlflow, mlflow.sklearn
from mlflow.models.signature import infer_signature
try:
    from sagemaker import Session
except ImportError:
    from sagemaker.session import Session                       # sagemaker >= 3
try:
    from sagemaker import get_execution_role
except ImportError:
    from sagemaker.session import get_execution_role            # sagemaker >= 3
from sagemaker.feature_store.feature_group      import FeatureGroup
from sagemaker.feature_store.feature_definition import (FeatureDefinition,
                                                        FeatureTypeEnum)
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("Imports OK")

In [ ]:
# ── CELL 3 : Configuration ────────────────────────────────────────────────────
ARTIFACT_BUCKET     = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
GOLD_BUCKET         = "cubic-mars-pm-s3-datalake-dev-gold-170202974600"
GOLD_PREFIX         = "chicago/gold/device_ps5_component"
GOLD_S3             = f"s3://{GOLD_BUCKET}/{GOLD_PREFIX}"

SM_SESSION          = Session()
ROLE                = get_execution_role()
REGION              = boto3.Session().region_name
SM_CLIENT           = boto3.client("sagemaker", region_name=REGION)
S3_CLIENT           = boto3.client("s3",         region_name=REGION)

MLFLOW_SERVER_ARN = "arn:aws:sagemaker:us-east-1:170202974600:mlflow-tracking-server/cubic-mars-mlflow-server-dev"
os.environ["MLFLOW_TRACKING_SERVER_ARN"] = MLFLOW_SERVER_ARN

try:
    MLFLOW_TRACKING_URI = SM_CLIENT.describe_mlflow_tracking_server(
        TrackingServerName="cubic-mars-mlflow-server-dev"
    )["TrackingServerUrl"]
    print(f"MLflow managed server: {MLFLOW_TRACKING_URI}")
except Exception:
    MLFLOW_TRACKING_URI = "file:///home/sagemaker-user/mlruns"
    print("MLflow managed server not found — using local file fallback (EFS)")
mlflow.set_tracking_uri(MLFLOW_SERVER_ARN)  # sagemaker-mlflow resolves arn: scheme

EXPERIMENT_NAME     = "chicago-ps5-remaining-useful-life"
MLFLOW_MODEL_NAME   = "ps5-remaining-useful-life"
MODEL_REGISTRY_NAME = "chicago-ps5-rul"
FS_GROUP_NAME       = "chicago-ps5-features"
ENDPOINT_NAME       = "chicago-ps5-rul-v1"

DURATION_COL        = "days_to_failure"   # event time (observed or censored)
EVENT_COL           = "is_not_censored"   # 1=failed, 0=censored (right-censored)
# READER removed 2026-06-24: reader is a component, not a standalone device category
# mars_device_category values: TVM / GATE / VALIDATOR / OTHER
DEVICE_TYPES        = ["TVM","GATE","VALIDATOR"]

mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Gold S3: {GOLD_S3}")

In [ ]:
# ── CELL 4 : FEASIBILITY WARNING ─────────────────────────────────────────────
print("!"*60)
print("PS5 FEASIBILITY: 50%")
print("  Gap 1: No work-order (WO) data in scope.")
print("         PM/CM intervals, part replacements = unavailable.")
print("  Gap 2: No PM records for GATE/READER/VALIDATOR.")
print("         TVM partial via SERVICENOW_AVAILABILITY_EVENTS only.")
print("  Gap 3: Component-level failure timestamps derived from")
print("         last observed event_dtm — may over-estimate RUL.")
print("  Impact: Survival models trained on degraded labels.")
print("          C-index ceiling ~0.65 (expect 0.55–0.62 without WO data)")
print("  Fix: Cubic to enable WO Oracle table export or ServiceNow API.")
print("!"*60)

In [ ]:
# ── CELL 5 : Load Gold table (Parquet on S3) ──────────────────────────────────
# G05 source: mars_dev.gold.device_ps5_component (built in Databricks from S16 + S17)
# Grain: one row per DEVICE_ID per component period (TVM / GATE / VALIDATOR).

print(f"Loading Gold table from: {GOLD_S3}")
df = pd.read_parquet(GOLD_S3, storage_options={"client_kwargs": {"region_name": REGION}})
GOLD_VERSION = None   # Parquet has no version; use None for MLflow tags
DATA_SOURCE  = "parquet"

df["component_start_dt"] = pd.to_datetime(df["component_start_dt"])
if "failure_dt" in df.columns:
    df["failure_dt"] = pd.to_datetime(df["failure_dt"])

# Gold produces mars_device_category; alias to device_type for downstream cells
if "mars_device_category" in df.columns and "device_type" not in df.columns:
    df = df.rename(columns={"mars_device_category": "device_type"})

print(f"Parquet source    : confirmed")
print(f"Shape             : {df.shape}")
print(f"\nCensoring summary:")
print(f"  Observed failures : {df[EVENT_COL].sum():,}  ({df[EVENT_COL].mean():.1%})")
print(f"  Censored          : {(df[EVENT_COL]==0).sum():,}  ({(df[EVENT_COL]==0).mean():.1%})")
print(f"\nDuration stats (days):")
print(df[DURATION_COL].describe())
print(f"\nDevice mix:\n{df['device_type'].value_counts()}")

In [ ]:
# ── CELL 6 : EDA — KM curves per device type ─────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=False)
axes = axes.flatten()

kmf = KaplanMeierFitter()
for i, dt_name in enumerate(DEVICE_TYPES):
    sub = df[df["device_type"]==dt_name] if "device_type" in df.columns else df
    if len(sub) < 10:
        axes[i].text(0.5,0.5,"No data",ha="center",va="center",transform=axes[i].transAxes)
        axes[i].set_title(f"KM — {dt_name}")
        continue
    T = sub[DURATION_COL].values
    E = sub[EVENT_COL].values
    kmf.fit(T, E, label=dt_name)
    kmf.plot_survival_function(ax=axes[i], ci_show=True, color="#1565C0")
    axes[i].set_title(f"KM Survival Curve — {dt_name}  (n={len(sub):,})")
    axes[i].set_xlabel("Days"); axes[i].set_ylabel("P(survival)")
    axes[i].axhline(0.5, color="red", linestyle="--", linewidth=0.8, label="Median RUL")
    axes[i].legend()

plt.suptitle("PS5 Kaplan–Meier Survival Curves by Device Type", fontsize=13)
plt.tight_layout(); plt.savefig("/tmp/ps5_km_curves.png", dpi=100); plt.show()

In [ ]:
# ── CELL 7 : Feature engineering & Feature Store ──────────────────────────────
ID_COLS = ["DEVICE_ID","component_start_dt","failure_dt","split"]
ID_COLS = [c for c in ID_COLS if c in df.columns]

FEATURE_COLS = [
    c for c in df.columns
    if c not in ID_COLS + [DURATION_COL, EVENT_COL, "is_censored", "device_type",
                            "device_id_hash"]
    and pd.api.types.is_numeric_dtype(df[c])
]
df[FEATURE_COLS] = df[FEATURE_COLS].fillna(df[FEATURE_COLS].median())
print(f"Features: {len(FEATURE_COLS)}")

# Temporal split: last 6 months = test, prior 6 months = val, rest = train
if "component_start_dt" in df.columns:
    max_dt = df["component_start_dt"].max()
    train_df = df[df["component_start_dt"] < max_dt - pd.Timedelta(days=180)]
    val_df   = df[(df["component_start_dt"] >= max_dt-pd.Timedelta(days=180)) &
                  (df["component_start_dt"] <  max_dt-pd.Timedelta(days=90))]
    test_df  = df[df["component_start_dt"] >= max_dt-pd.Timedelta(days=90)]
else:
    n = len(df)
    train_df = df.iloc[:int(n*0.7)]
    val_df   = df.iloc[int(n*0.7):int(n*0.85)]
    test_df  = df.iloc[int(n*0.85):]

print(f"Train:{len(train_df):,}  Val:{len(val_df):,}  Test:{len(test_df):,}")

# Feature Store
fs_df = train_df[FEATURE_COLS+[DURATION_COL,EVENT_COL,"DEVICE_ID"]].copy()
if "component_start_dt" in train_df.columns:
    fs_df["event_time"] = train_df["component_start_dt"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
else:
    fs_df["event_time"] = "2024-01-01T00:00:00Z"
fs_df["DEVICE_ID"] = fs_df["DEVICE_ID"].astype(str)

def dtype_to_fs(dtype):
    if pd.api.types.is_float_dtype(dtype):   return FeatureTypeEnum.FRACTIONAL
    if pd.api.types.is_integer_dtype(dtype): return FeatureTypeEnum.INTEGRAL
    return FeatureTypeEnum.STRING

feature_group = FeatureGroup(name=FS_GROUP_NAME, sagemaker_session=SM_SESSION,
    feature_definitions=[
        FeatureDefinition(feature_name=c, feature_type=dtype_to_fs(fs_df[c].dtype))
        for c in fs_df.columns])
try:
    feature_group.create(
        s3_uri=f"s3://{ARTIFACT_BUCKET}/feature-store/{FS_GROUP_NAME}",
        record_identifier_name="DEVICE_ID", event_time_feature_name="event_time",
        role_arn=ROLE, enable_online_store=True,
        description="PS5 RUL survival features — Chicago Ventra")
    status=""
    while status != "Created":
        time.sleep(15)
        status = feature_group.describe().get("FeatureGroupStatus")
        print(f"  {status}")
except Exception as e: print(f"Feature Group: {e}")
feature_group.ingest(data_frame=fs_df.reset_index(drop=True), max_workers=4, wait=True)
print(f"Ingested {len(fs_df):,} → {FS_GROUP_NAME}")

In [ ]:
# ── CELL 8 : Helpers ──────────────────────────────────────────────────────────
COMMON_TAGS = {"ps":"PS5","target":"days_to_failure","city":"chicago",
               "gold_version":str(GOLD_VERSION),"architecture":"CUBIC-MARS-AWS-v6",
               "feasibility":"50pct","primary_metric":"concordance_index"}

def c_index(model_obj, T, E, X=None, model_type="lifelines"):
    """Compute concordance index — primary metric for PS5."""
    if model_type == "lifelines":
        pred = model_obj.predict_median(X) if X is not None else model_obj.predict_median()
        pred = pred.fillna(pred.median())
        return round(concordance_index(T, pred, E), 4)
    elif model_type == "sksurv":
        return round(model_obj.score(X, Surv.from_arrays(E.astype(bool), T)), 4)
    elif model_type == "sklearn":
        pred = model_obj.predict(X)
        return round(concordance_index(T, pred, E), 4)

In [ ]:
# ── CELL 9 : MODEL 1 — Weibull AFT (parametric baseline) ─────────────────────
# WeibullAFTFitter: accelerated failure time, log-normal hazard
# Most interpretable — survival S(t) closed-form, usable for regulatory docs

aft_df_train = train_df[FEATURE_COLS + [DURATION_COL, EVENT_COL]].copy()
aft_df_val   = val_df[FEATURE_COLS + [DURATION_COL, EVENT_COL]].copy()
aft_df_test  = test_df[FEATURE_COLS + [DURATION_COL, EVENT_COL]].copy()

with mlflow.start_run(run_name="weibull_aft") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type":"weibull_aft_lifelines"})
    waf = WeibullAFTFitter(penalizer=0.05, l1_ratio=0.1)
    waf.fit(aft_df_train, duration_col=DURATION_COL, event_col=EVENT_COL)

    ci_val  = c_index(waf, val_df[DURATION_COL].values,
                      val_df[EVENT_COL].values, aft_df_val)
    ci_test = c_index(waf, test_df[DURATION_COL].values,
                      test_df[EVENT_COL].values, aft_df_test)

    mlflow.log_params({"penalizer":0.05,"l1_ratio":0.1,"model":"WeibullAFT",
                       "feasibility_note":"50pct_no_WO_data"})
    mlflow.log_metrics({"val_cindex":ci_val,"test_cindex":ci_test})

    # Log Weibull param summary
    with open("/tmp/weibull_summary.txt","w") as f:
        f.write(waf.summary.to_string())
    mlflow.log_artifact("/tmp/weibull_summary.txt")

    # Survival curve plot (average device)
    fig,ax=plt.subplots(figsize=(9,5))
    waf.predict_survival_function(aft_df_test.iloc[:5]).T.plot(ax=ax)
    ax.set_title("Weibull AFT — Survival Curves (5 sample devices)")
    ax.set_xlabel("Days"); ax.set_ylabel("S(t) — P(survival)")
    fig.tight_layout(); fig.savefig("/tmp/ps5_weibull_curves.png",dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/ps5_weibull_curves.png")

    joblib.dump(waf,"/tmp/ps5_weibull_aft.joblib")
    mlflow.log_artifact("/tmp/ps5_weibull_aft.joblib","model")
    WAF_RUN=run.info.run_id

print(f"Weibull AFT  val_ci={ci_val}  test_ci={ci_test}")

In [ ]:
# ── CELL 10 : MODEL 2 — Cox Proportional Hazards ─────────────────────────────
# CoxPHFitter: semi-parametric, no parametric baseline assumption
# Hazard ratio interpretation for each covariate

with mlflow.start_run(run_name="cox_ph") as run:
    mlflow.set_tags({**COMMON_TAGS, "model_type":"cox_ph_lifelines"})
    cph = CoxPHFitter(penalizer=0.1, l1_ratio=0.0)
    cph.fit(aft_df_train, duration_col=DURATION_COL, event_col=EVENT_COL)

    ci_val  = c_index(cph, val_df[DURATION_COL].values,
                      val_df[EVENT_COL].values, aft_df_val)
    ci_test = c_index(cph, test_df[DURATION_COL].values,
                      test_df[EVENT_COL].values, aft_df_test)

    mlflow.log_params({"penalizer":0.1,"l1_ratio":0.0,"model":"CoxPH"})
    mlflow.log_metrics({"val_cindex":ci_val,"test_cindex":ci_test})

    # Top hazard ratios
    hr_df = np.exp(cph.params_).rename("hazard_ratio").sort_values(ascending=False)
    with open("/tmp/cox_hazard_ratios.csv","w") as f:
        hr_df.to_csv(f)
    mlflow.log_artifact("/tmp/cox_hazard_ratios.csv")

    # Cox log-log check
    try:
        fig,ax=plt.subplots(figsize=(9,5))
        cph.check_assumptions(aft_df_train,p_value_threshold=0.05,plot_n=3,
                               show_plots=False)
        fig.savefig("/tmp/ps5_cox_assumptions.png",dpi=100); plt.close(fig)
        mlflow.log_artifact("/tmp/ps5_cox_assumptions.png")
    except Exception: pass

    joblib.dump(cph,"/tmp/ps5_cox_ph.joblib")
    mlflow.log_artifact("/tmp/ps5_cox_ph.joblib","model")
    CPH_RUN=run.info.run_id

print(f"Cox PH  val_ci={ci_val}  test_ci={ci_test}")

In [ ]:
# ── CELL 11 : MODEL 3 — Random Survival Forest (scikit-survival) ──────────────
# RSF: non-parametric tree ensemble for survival data
# Handles interactions + non-linearity better than Cox PH
# Requires structured array y = [(is_not_censored, days_to_failure)]

scaler_rsf = StandardScaler()
X_train_rsf = scaler_rsf.fit_transform(train_df[FEATURE_COLS].values)
X_val_rsf   = scaler_rsf.transform(val_df[FEATURE_COLS].values)
X_test_rsf  = scaler_rsf.transform(test_df[FEATURE_COLS].values)

y_train_rsf = Surv.from_arrays(train_df[EVENT_COL].astype(bool).values,
                                train_df[DURATION_COL].values)
y_test_rsf  = Surv.from_arrays(test_df[EVENT_COL].astype(bool).values,
                                test_df[DURATION_COL].values)
y_val_rsf   = Surv.from_arrays(val_df[EVENT_COL].astype(bool).values,
                                val_df[DURATION_COL].values)

with mlflow.start_run(run_name="random_survival_forest") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"random_survival_forest_sksurv"})

    rsf = RandomSurvivalForest(
        n_estimators      = 200,
        min_samples_split = 20,
        min_samples_leaf  = 15,
        max_features      = "sqrt",
        n_jobs            = -1,
        random_state      = 42,
    )
    rsf.fit(X_train_rsf, y_train_rsf)

    ci_val  = round(rsf.score(X_val_rsf,  y_val_rsf),  4)
    ci_test = round(rsf.score(X_test_rsf, y_test_rsf), 4)

    mlflow.log_params({"n_estimators":200,"min_samples_split":20,
                       "min_samples_leaf":15,"max_features":"sqrt"})
    mlflow.log_metrics({"val_cindex":ci_val,"test_cindex":ci_test})

    # SHAP-based feature importance (tree models)
    try:
        explainer = shap.TreeExplainer(rsf)
        sv = explainer(pd.DataFrame(X_test_rsf[:500],columns=FEATURE_COLS))
        shap.plots.bar(sv,max_display=20,show=False)
        fig = plt.gcf(); ax = plt.gca()
        ax.set_title("PS5 RSF — SHAP Feature Importance",pad=12)
        fig.tight_layout(); fig.savefig("/tmp/ps5_rsf_shap.png",dpi=100); plt.close(fig)
        mlflow.log_artifact("/tmp/ps5_rsf_shap.png")
    except Exception: pass

    joblib.dump(rsf,       "/tmp/ps5_rsf.joblib")
    joblib.dump(scaler_rsf,"/tmp/ps5_rsf_scaler.joblib")
    mlflow.log_artifact("/tmp/ps5_rsf.joblib",       "model")
    mlflow.log_artifact("/tmp/ps5_rsf_scaler.joblib","model")
    RSF_RUN=run.info.run_id

print(f"RSF  val_ci={ci_val}  test_ci={ci_test}")

In [ ]:
# ── CELL 12 : MODEL 4 — Kaplan–Meier per device type (non-parametric) ─────────
# KM for group-level survival curves — used in operational reporting.
# Logged as fitted objects + survival function tables.

km_results={}
with mlflow.start_run(run_name="kaplan_meier_by_device") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"kaplan_meier_device_type"})
    naf=NelsonAalenFitter()
    fig,axes=plt.subplots(2,2,figsize=(14,10))
    axes=axes.flatten()

    for i,dt_name in enumerate(DEVICE_TYPES):
        sub = (df[df["device_type"]==dt_name]
               if "device_type" in df.columns else df)
        if len(sub) < 5:
            axes[i].text(0.5,0.5,"Insufficient data\n(expected after WO data added)",
                         ha="center",va="center",fontsize=9,transform=axes[i].transAxes)
            axes[i].set_title(f"KM — {dt_name}")
            continue
        T=sub[DURATION_COL].values; E=sub[EVENT_COL].values
        kmf_dt=KaplanMeierFitter(label=dt_name)
        kmf_dt.fit(T,E)
        kmf_dt.plot_survival_function(ax=axes[i],ci_show=True)
        med_rul=kmf_dt.median_survival_time_
        axes[i].set_title(f"KM — {dt_name}  (median RUL={med_rul:.0f}d)")
        axes[i].axhline(0.5,color="red",linestyle="--",linewidth=0.8)
        km_results[dt_name]={"median_rul_days":float(med_rul),"n":len(sub),
                              "event_rate":float(E.mean())}
        mlflow.log_metric(f"{dt_name}_median_rul_days",float(med_rul))
        mlflow.log_metric(f"{dt_name}_n",               len(sub))

        joblib.dump(kmf_dt,f"/tmp/ps5_km_{dt_name}.joblib")
        mlflow.log_artifact(f"/tmp/ps5_km_{dt_name}.joblib","km_models")

    plt.suptitle("KM Survival Curves by Device Type",fontsize=13)
    plt.tight_layout(); plt.savefig("/tmp/ps5_km_all.png",dpi=100); plt.close(fig)
    mlflow.log_artifact("/tmp/ps5_km_all.png")

    with open("/tmp/ps5_km_results.json","w") as f: json.dump(km_results,f,indent=2)
    mlflow.log_artifact("/tmp/ps5_km_results.json")
    KM_RUN=run.info.run_id

print("KM fitted per device type:")
for k,v in km_results.items():
    print(f"  {k}: median_RUL={v.get('median_rul_days','N/A'):.0f}d  n={v.get('n',0):,}")

In [ ]:
# ── CELL 13 : MODEL 5 — GBM regressor on uncensored subset ────────────────────
# Direct regression on time-to-failure for uncensored observations only.
# Useful for short-range (< 30d) operational alerts.

unc_train = train_df[train_df[EVENT_COL]==1]
unc_test  = test_df[test_df[EVENT_COL]==1]

if len(unc_train) >= 50:
    X_unc_tr = unc_train[FEATURE_COLS].values
    y_unc_tr = unc_train[DURATION_COL].values
    X_unc_te = unc_test[FEATURE_COLS].values
    y_unc_te = unc_test[DURATION_COL].values

    with mlflow.start_run(run_name="gbm_regression_uncensored") as run:
        mlflow.set_tags({**COMMON_TAGS,"model_type":"gbm_regression_uncensored",
                         "note":"uncensored_only_subset"})
        gbr=GradientBoostingRegressor(n_estimators=300,max_depth=5,
                                       learning_rate=0.05,subsample=0.8,
                                       random_state=42)
        gbr.fit(X_unc_tr,y_unc_tr)
        pred_te=gbr.predict(X_unc_te)
        mae=mean_absolute_error(y_unc_te,pred_te)
        ci=concordance_index(y_unc_te,pred_te)
        mlflow.log_params({"n_estimators":300,"max_depth":5,"subset":"uncensored"})
        mlflow.log_metrics({"test_mae_days":round(mae,2),"test_cindex":round(ci,4)})
        sig=infer_signature(X_unc_tr,gbr.predict(X_unc_tr))
        mlflow.sklearn.log_model(gbr,"model",signature=sig)
        GBR_RUN=run.info.run_id
    print(f"GBM regression (uncensored)  MAE={mae:.1f}d  C-index={ci:.4f}")
else:
    print(f"Uncensored train set too small ({len(unc_train)}) — GBM regression skipped")
    GBR_RUN=None

In [ ]:
# ── CELL 14 : Optuna — tune RSF (all-fold C-index) ───────────────────────────
TS_CV=TimeSeriesSplit(n_splits=5)
all_idx=np.arange(len(X_train_rsf)+len(X_val_rsf))
X_cv_rsf=np.vstack([X_train_rsf,X_val_rsf])
# Safe: use named fields so both sides have guaranteed matching dtype
y_cv_rsf = Surv.from_arrays(
    event = np.concatenate([y_train_rsf['e'], y_val_rsf['e']]),
    time  = np.concatenate([y_train_rsf['t'], y_val_rsf['t']])
)

def optuna_rsf(trial):
    p=dict(
        n_estimators     =trial.suggest_int("n_estimators",100,400),
        min_samples_split=trial.suggest_int("min_samples_split",5,40),
        min_samples_leaf =trial.suggest_int("min_samples_leaf",5,30),
        max_features     =trial.suggest_categorical("max_features",["sqrt","log2",0.5]),
        n_jobs=-1,random_state=42)
    scores=[]
    for step,(tr,va) in enumerate(TS_CV.split(X_cv_rsf)):
        m=RandomSurvivalForest(**p)
        m.fit(X_cv_rsf[tr],y_cv_rsf[tr])
        scores.append(m.score(X_cv_rsf[va],y_cv_rsf[va]))
        trial.report(np.mean(scores),step)
        if trial.should_prune(): raise optuna.TrialPruned()
    return np.mean(scores)

study_rsf=optuna.create_study(direction="maximize",
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5))
study_rsf.optimize(optuna_rsf,n_trials=30,show_progress_bar=True)
BEST_RSF={**study_rsf.best_params,"n_jobs":-1,"random_state":42}
print(f"Best RSF C-index (Optuna): {study_rsf.best_value:.4f}")
print(f"Best params: {BEST_RSF}")

In [ ]:
# ── CELL 15 : RSF tuned — champion candidate ─────────────────────────────────
with mlflow.start_run(run_name="rsf_optuna_tuned") as run:
    mlflow.set_tags({**COMMON_TAGS,"model_type":"rsf_optuna_tuned"})
    rsf_tuned=RandomSurvivalForest(**BEST_RSF)
    rsf_tuned.fit(X_cv_rsf,y_cv_rsf)
    ci_val  =round(rsf_tuned.score(X_val_rsf,  y_val_rsf), 4)
    ci_test =round(rsf_tuned.score(X_test_rsf, y_test_rsf),4)
    mlflow.log_params({**BEST_RSF,"optuna_cv":round(study_rsf.best_value,4)})
    mlflow.log_metrics({"val_cindex":ci_val,"test_cindex":ci_test})
    joblib.dump(rsf_tuned,"/tmp/ps5_rsf_tuned.joblib")
    mlflow.log_artifact("/tmp/ps5_rsf_tuned.joblib","model")
    RSF_TUNED_RUN=run.info.run_id
print(f"RSF(Optuna) val_ci={ci_val}  test_ci={ci_test}")

In [ ]:
# ── CELL 16 : Select champion + register ─────────────────────────────────────
# Select model with highest test_cindex across RSF, CoxPH, WeibullAFT runs
all_runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME],
                               order_by=["metrics.test_cindex DESC"])
candidate_names = ["weibull_aft","cox_ph","random_survival_forest",
                   "rsf_optuna_tuned","gbm_regression_uncensored"]
cand_mask = all_runs["tags.mlflow.runName"].isin(candidate_names)
champion_row  = all_runs[cand_mask].iloc[0]
CHAMPION_RUN_ID = champion_row["run_id"]
CHAMPION_NAME   = champion_row["tags.mlflow.runName"]
CHAMPION_CI     = champion_row["metrics.test_cindex"]
print(f"Champion: {CHAMPION_NAME}  C-index={CHAMPION_CI:.4f}")

mlfc=mlflow.tracking.MlflowClient()
try: mlfc.create_registered_model(MLFLOW_MODEL_NAME,
        description="PS5 RUL Survival Analysis — Chicago Ventra (feasibility=50%)")
except: pass
mv=mlfc.create_model_version(name=MLFLOW_MODEL_NAME,
    source=f"runs:/{CHAMPION_RUN_ID}/model",run_id=CHAMPION_RUN_ID,
    tags={"algorithm":CHAMPION_NAME,"test_cindex":str(CHAMPION_CI),
          "gold_version":str(GOLD_VERSION),"feasibility":"50pct"})
mlfc.transition_model_version_stage(MLFLOW_MODEL_NAME,mv.version,"Staging")
MLFLOW_VERSION=mv.version
print(f"MLflow: {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION} → Staging")

# Upload all survival models to S3
artifacts={
    "weibull_aft": "/tmp/ps5_weibull_aft.joblib",
    "cox_ph":      "/tmp/ps5_cox_ph.joblib",
    "rsf_tuned":   "/tmp/ps5_rsf_tuned.joblib",
    "rsf_scaler":  "/tmp/ps5_rsf_scaler.joblib",
    "km_results":  "/tmp/ps5_km_results.json",
}
for key,path in artifacts.items():
    if os.path.exists(path):
        S3_CLIENT.upload_file(path,ARTIFACT_BUCKET,
                              f"sagemaker/ps5/model-v{MLFLOW_VERSION}/{key}")

# Package champion for SageMaker
with tarfile.open("/tmp/model.tar.gz","w:gz") as tar:
    for key,path in artifacts.items():
        if os.path.exists(path): tar.add(path,arcname=os.path.basename(path))
MODEL_S3_KEY=f"sagemaker/ps5/model-v{MLFLOW_VERSION}/model.tar.gz"
S3_CLIENT.upload_file("/tmp/model.tar.gz",ARTIFACT_BUCKET,MODEL_S3_KEY)

import sagemaker.image_uris as sm_images
image_uri=sm_images.retrieve(
    framework="sklearn", region=REGION, version="1.2-1",
    instance_type="ml.m5.large", image_scope="inference")
try: SM_CLIENT.create_model_package_group(ModelPackageGroupName=MODEL_REGISTRY_NAME,
        ModelPackageGroupDescription="PS5 RUL — Chicago Ventra")
except: pass
resp=SM_CLIENT.create_model_package(
    ModelPackageGroupName=MODEL_REGISTRY_NAME,
    ModelPackageDescription=f"{CHAMPION_NAME}|CI={CHAMPION_CI:.4f}|Gv{GOLD_VERSION}|feasibility=50%",
    InferenceSpecification={
        "Containers":[{"Image":image_uri,"ModelDataUrl":f"s3://{ARTIFACT_BUCKET}/{MODEL_S3_KEY}"}],
        "SupportedContentTypes":["text/csv"],"SupportedResponseMIMETypes":["application/json"],
        "SupportedRealtimeInferenceInstanceTypes":["ml.m5.large"]},
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={"ps":"PS5","algorithm":CHAMPION_NAME,
        "test_cindex":str(round(CHAMPION_CI,4)),"mlflow_version":str(MLFLOW_VERSION),
        "feasibility":"50pct_no_WO_data"})
SM_MODEL_PKG_ARN=resp["ModelPackageArn"]
print(f"SM Package: {SM_MODEL_PKG_ARN}")

In [ ]:
# ── CELL 17 : Deploy + Model Monitor ─────────────────────────────────────────
from sagemaker.model import ModelPackage
from sagemaker.model_monitor import (DefaultModelMonitor, DataCaptureConfig,
                                      CronExpressionGenerator)
from sagemaker.model_monitor.dataset_format import DatasetFormat

SM_CLIENT.update_model_package(ModelPackageArn=SM_MODEL_PKG_ARN,
                                ModelApprovalStatus="Approved")
registered=ModelPackage(role=ROLE,model_package_arn=SM_MODEL_PKG_ARN,
                         sagemaker_session=SM_SESSION)
predictor=registered.deploy(
    initial_instance_count=1, instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME,
    data_capture_config=DataCaptureConfig(
        enable_capture=True, sampling_percentage=100,
        destination_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps5/data-capture/",
        capture_options=["Input","Output"]))
print(f"Endpoint: {ENDPOINT_NAME}")

monitor=DefaultModelMonitor(role=ROLE,instance_count=1,
    instance_type="ml.m5.large",sagemaker_session=SM_SESSION)
train_df[FEATURE_COLS].to_csv("/tmp/ps5_baseline.csv",index=False)
S3_CLIENT.upload_file("/tmp/ps5_baseline.csv",ARTIFACT_BUCKET,
                      "sagemaker/ps5/baseline/baseline.csv")
monitor.suggest_baseline(
    baseline_dataset=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps5/baseline/baseline.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps5/baseline/output/",
    wait=True,logs=False)
# PS5 is component-grain — weekly refresh aligns with component inspection cycle
monitor.create_monitoring_schedule(
    monitor_schedule_name="chicago-ps5-data-quality",endpoint_input=ENDPOINT_NAME,
    output_s3_uri=f"s3://{ARTIFACT_BUCKET}/sagemaker/ps5/monitor-output/",
    statistics=monitor.baseline_statistics(),constraints=monitor.suggested_constraints(),
    schedule_cron_expression="cron(0 6 ? * MON *)",   # weekly Monday 06:00 UTC
    enable_cloudwatch_metrics=True)

cw=boto3.client("cloudwatch",region_name=REGION)
cw.put_metric_alarm(AlarmName="PS5-DataQuality-ViolationRate",
    ComparisonOperator="GreaterThanThreshold",EvaluationPeriods=1,
    MetricName="feature_baseline_drift_percentage",
    Namespace="aws/sagemaker/Endpoints/data-metrics",
    Period=604800,Statistic="Average",Threshold=3.0,
    AlarmDescription="PS5 RUL feature violation >3% — CUBIC MARS 97% gate",
    Dimensions=[{"Name":"Endpoint","Value":ENDPOINT_NAME}],
    TreatMissingData="notBreaching")
print("Monitor (weekly) + alarm configured.")

In [ ]:
# ── CELL 18 : Ground truth loop — C-index tracking over time ─────────────────
# After observed failures accumulate (weekly cadence), upload actual outcomes.
# Model Quality Monitor computes C-index on ground truth vs predicted RUL.

GROUND_TRUTH_UPLOAD_CODE = '''
import boto3, json, pandas as pd, datetime

ARTIFACT_BUCKET = "cubic-mars-pm-s3-datalake-dev-artifacts-170202974600"
ENDPOINT_NAME   = "chicago-ps5-rul-v1"

def upload_ps5_ground_truth(actual_failures_df: pd.DataFrame, run_date: str):
    """
    actual_failures_df columns:
        DEVICE_ID, actual_days_to_failure, event_observed (1=failed, 0=censored)
    run_date: "YYYY-MM-DD" — Monday of each week
    """
    lines=[]
    for _,row in actual_failures_df.iterrows():
        lines.append(json.dumps({
            "groundTruthData":{
                "data":str(row["actual_days_to_failure"]),
                "encoding":"CSV"
            },
            "eventMetadata":{
                "eventId":str(row["DEVICE_ID"]),
                "inferenceTime":run_date+"T00:00:00Z"
            },
            "eventVersion":"0"
        }))
    yr,mo,dy=run_date.split("-")
    key=f"sagemaker/ps5/ground-truth/{yr}/{mo}/{dy}/gt.jsonl"
    boto3.client("s3").put_object(
        Bucket=ARTIFACT_BUCKET, Key=key,
        Body="\\n".join(lines), ContentType="application/jsonlines")
    print(f"Ground truth uploaded: s3://{ARTIFACT_BUCKET}/{key}  ({len(lines)} records)")
'''
print("Ground truth upload template (run weekly on Mondays after failure events):")
print(GROUND_TRUTH_UPLOAD_CODE)

S3_CLIENT.put_object(Bucket=ARTIFACT_BUCKET,
    Key="sagemaker/ps5/utils/upload_ground_truth.py",
    Body=GROUND_TRUTH_UPLOAD_CODE, ContentType="text/x-python")
print(f"\nUploaded to s3://{ARTIFACT_BUCKET}/sagemaker/ps5/utils/upload_ground_truth.py")

In [ ]:
# ── CELL 19 : Summary ────────────────────────────────────────────────────────
print("="*60)
print("PS5 — Remaining Useful Life — Complete")
print("="*60)
print(f"Champion      : {CHAMPION_NAME}  C-index={CHAMPION_CI:.4f}")
print(f"MLflow model  : {MLFLOW_MODEL_NAME} v{MLFLOW_VERSION}")
print(f"SM endpoint   : {ENDPOINT_NAME}")
print(f"Monitor freq  : weekly (component grain)")
print(f"Primary metric: concordance index (not AUC)")
print(f"\nKM median RUL by device type:")
for k,v in km_results.items():
    print(f"  {k}: {v.get('median_rul_days','N/A')} days")
print(f"\n{'!'*50}")
print(f"FEASIBILITY = 50%")
print(f"  Current C-index ceiling ~0.65 without WO data")
print(f"  Models: WeibullAFT | CoxPH | RSF | KM | GBM(uncensored)")
print(f"  Blocked by: no WO table, no PM for GATE/READER/VALIDATOR")
print(f"  Fix: Enable Cubic WO Oracle export OR ServiceNow API")
print(f"       Estimated gain: 50% → 85% feasibility after WO data added")
print(f"{'!'*50}")